# Préparation Finale — Features & Cibles
Entrée : `features_encodees.parquet` → Sortie : `X.parquet` + `y.parquet`

- Sélection des 5 cibles `out.*`
- Transformation logarithmique (log1p) des colonnes asymétriques
- Gestion des NaN résiduels
- Séparation features / cibles

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'

df = pd.read_parquet(DATA_PROCESSED / 'features_encodees.parquet')
print('Dataset chargé :', df.shape)

Dataset chargé : (549971, 415)


## 1. Sélection des cibles

In [8]:
TARGETS = [
    'out.electricity.total.energy_consumption..kwh',
    'out.natural_gas.total.energy_consumption..kwh',
    'out.fuel_oil.total.energy_consumption..kwh',
    'out.propane.total.energy_consumption..kwh',
    'out.emissions.total.lrmer_mid_case_25..co2e_kg',
]

TARGETS = [c for c in TARGETS if c in df.columns]

out_all  = [c for c in df.columns if c.startswith('out.')]
out_drop = [c for c in out_all if c not in TARGETS]

df.drop(columns=out_drop, inplace=True)
print(f'Cibles conservées : {len(TARGETS)}')

Cibles conservées : 5


## 3. Gestion des NaN résiduels
- Colonnes > 30 % NaN → supprimées
- NaN restants (numériques) → médiane
- Lignes encore NaN → supprimées

In [9]:
nan_pct = df.isna().mean()

drop_nan_cols = nan_pct[nan_pct > 0.30].index.tolist()
df.drop(columns=drop_nan_cols, inplace=True)
print(f'Colonnes supprimées (>30% NaN) : {len(drop_nan_cols)}')

nan_pct  = df.isna().mean()
num_cols = df.select_dtypes(include=[np.number]).columns
medians  = df[num_cols].median()
fill_cols = nan_pct[nan_pct > 0].index.intersection(num_cols)
df[fill_cols] = df[fill_cols].fillna(medians[fill_cols])
print(f'Colonnes imputées (médiane)    : {len(fill_cols)}')

rows_before = len(df)
df.dropna(inplace=True)
print(f'Lignes supprimées (NaN restants): {rows_before - len(df)}')

Colonnes supprimées (>30% NaN) : 18
Colonnes imputées (médiane)    : 6
Lignes supprimées (NaN restants): 0


In [10]:
df = pd.read_parquet(DATA_PROCESSED / 'X.parquet')
print("Chargé :", df.shape)

#Colonnes string / object résiduelles 
str_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
print(f"\nColonnes string à supprimer ({len(str_cols)}) :")
print(str_cols)
df.drop(columns=str_cols, inplace=True)

#Colonnes entièrement vides (100 % NaN) 
empty_cols = df.columns[df.isna().all()].tolist()
print(f"\nColonnes vides ({len(empty_cols)}) : {empty_cols}")
df.drop(columns=empty_cols, inplace=True)

#Colonnes quasi-constantes (variance ≈ 0 ou unique value > 99.5 %) 
quasi_const = []
for col in df.columns:
    top_freq = df[col].value_counts(normalize=True, dropna=False).iloc[0]
    if top_freq >= 0.995:
        quasi_const.append(col)

print(f"\nColonnes quasi-constantes ({len(quasi_const)}) :")
print(quasi_const)
df.drop(columns=quasi_const, inplace=True)

# Colonnes dupliquées (valeurs identiques)
dup_cols = []
seen = {}
for col in df.columns:
    key = tuple(df[col].values[:500])   # empreinte rapide sur les 500 premières lignes
    if key in seen:
        dup_cols.append(col)
    else:
        seen[key] = col

print(f"\nColonnes dupliquées ({len(dup_cols)}) : {dup_cols}")
df.drop(columns=dup_cols, inplace=True)

#Résumé
print(f"\n df final : {df.shape}")
print(df.dtypes.value_counts())


Chargé : (549971, 94)

Colonnes string à supprimer (21) :
['in.ceiling_fan', 'in.census_region', 'in.clothes_dryer', 'in.clothes_washer', 'in.clothes_washer_presence', 'in.cooking_range', 'in.dishwasher', 'in.duct_leakage_and_insulation', 'in.duct_location', 'in.lighting', 'in.misc_extra_refrigerator', 'in.misc_freezer', 'in.misc_gas_fireplace', 'in.misc_gas_grill', 'in.misc_gas_lighting', 'in.misc_hot_tub_spa', 'in.misc_pool', 'in.misc_pool_heater', 'in.misc_pool_pump', 'in.misc_well_pump', 'in.refrigerator']

Colonnes vides (0) : []

Colonnes quasi-constantes (8) :
['in.hvac_secondary_heating_partial_space_conditioning', 'in.lighting_interior_use', 'in.lighting_other_use', 'in.solar_collector_area', 'in.water_heater_technology_solar', 'in.water_heater_fuel_other', 'in.water_heater_fuel_solar', 'in.water_heater_fuel_wood']

Colonnes dupliquées (1) : ['in.insulation_rim_joist']

 df final : (549971, 64)
float64    24
int64      15
Int8       14
float32    11
Name: count, dtype: int64


## 4. Séparation X / y

In [13]:
print(df.columns.tolist())

['in.aiannh_area', 'in.area_median_income', 'in.bathroom_spot_vent_hour', 'in.bedrooms', 'in.cooling_unavailable_days', 'in.cooling_setpoint', 'in.cooling_setpoint_has_offset', 'in.cooling_setpoint_offset_magnitude', 'in.corridor', 'in.electric_panel_service_rating..a', 'in.electric_vehicle_charge_at_home', 'in.electric_vehicle_charger', 'in.electric_vehicle_miles_traveled', 'in.federal_poverty_level', 'in.geometry_attic_type', 'in.geometry_floor_area', 'in.geometry_foundation_type', 'in.geometry_garage', 'in.geometry_stories', 'in.ground_thermal_conductivity', 'in.heating_unavailable_days', 'in.heating_setpoint', 'in.heating_setpoint_has_offset', 'in.heating_setpoint_offset_magnitude', 'in.hot_water_fixtures', 'in.household_has_tribal_persons', 'in.hvac_cooling_efficiency', 'in.hvac_cooling_partial_space_conditioning', 'in.hvac_has_ducts', 'in.hvac_has_zonal_electric_heating', 'in.income', 'in.insulation_ceiling', 'in.insulation_floor', 'in.insulation_foundation_wall', 'in.insulation_

In [12]:

X = df[[c for c in df.columns if c.startswith('in.')]]
y = df[TARGETS]

print(f'X : {X.shape}')
print(f'y : {y.shape}')



KeyError: "None of [Index(['out.electricity.total.energy_consumption..kwh',\n       'out.natural_gas.total.energy_consumption..kwh',\n       'out.fuel_oil.total.energy_consumption..kwh',\n       'out.propane.total.energy_consumption..kwh',\n       'out.emissions.total.lrmer_mid_case_25..co2e_kg'],\n      dtype='str')] are in the [columns]"

## 5. Sauvegarde → `X.parquet` + `y.parquet`

In [5]:
X.to_parquet(DATA_PROCESSED / 'X.parquet', index=False)
y.to_parquet(DATA_PROCESSED / 'y.parquet', index=False)

print(f'X sauvegardé : {DATA_PROCESSED}/X.parquet  {X.shape}')
print(f'y sauvegardé : {DATA_PROCESSED}/y.parquet  {y.shape}')

X sauvegardé : \\FS-SOP\Staff-CMA\yzouarhi\Bureau\Data\FlexiMax\data\processed/X.parquet  (549971, 94)
y sauvegardé : \\FS-SOP\Staff-CMA\yzouarhi\Bureau\Data\FlexiMax\data\processed/y.parquet  (549971, 5)
